In [1]:
# Import libraries

import warnings
# Pandas for data handling
import pandas # https://pandas.pydata.org/
from pandas.plotting import scatter_matrix
# from pandas.plotting import scatter_matrix

# pretty tables
from IPython.display import display

# NumPy for numerical computing
import numpy as np# https://numpy.org/

# MatPlotLib+Seaborn for visualization
import matplotlib.pyplot as pl  # https://matplotlib.org/
import seaborn as sns

# assessment
from sklearn import model_selection # for model comparisons
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn import model_selection # for model comparisons
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import cohen_kappa_score

# algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
# from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# data preprocessing / feature selection
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# combining
from sklearn.pipeline import make_pipeline

#########

In [2]:
print('Loading data from file ...')  # load the data
dataset = pandas.read_csv('winequality-white.csv')
print('done \n')

print('Removing rows with missing data ...')  # Remove missing data
dataset = dataset.dropna()  # default is to drop any row that contains at least one missing value
print('done \n')

Loading data from file ...
done 

Removing rows with missing data ...
done 



In [3]:
# Let's set up a problem: Can we predict 'callSign' using these three features: 'Depth', 'Temperature', 'Salinity' ?

print('Reading list of problem variables X and Y...')
X_name = [ 'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol' ] # columns to focus on as predictors
X = dataset[X_name]   # only keep these columns as features
y_name = 'quality'     # column to focus on as target
y = dataset[y_name]   # only keep this column as label 
print('done \n')

# Split-out test dataset

# We reset the random number seed before each run to ensure that the evaluation of each algorithm is performed using exactly the same data splits. It ensures the results are directly comparable.
seed = 42

# Train, test split
print('Partitioning data into parts: formative (for development) and summative (for testing) ...')
test_size = 0.20   # means 20 percent

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=test_size, random_state=seed)
print('done \n')

Reading list of problem variables X and Y...
done 

Partitioning data into parts: formative (for development) and summative (for testing) ...
done 



In [4]:
# EDA

# display(X_train.describe(include='all'))
# X_train.hist(figsize=(15, 15), bins=12)  # bins ~= sqrt(N)
# pl.show()

# display(y_train.describe(include='all'))

In [5]:
import seaborn as sns   # https://seaborn.pydata.org/

# print('Summary of X - Bivariate (column-pair) graphs:')

# print('Correlation matrix:')
# corr = X.corr()
# sns.heatmap( corr, cmap='coolwarm', vmax=1.0, vmin=-1.0 );
# pl.show()

In [6]:
# From the correlation matrix, we can see residual sugar is related to density. 
# Free sulfur dioxide is related to total sulfur dioxide. etc. 

In [7]:
#Classification problem: predicting the quality value (a single variable with 
#seven classes labeled 3, 4, 5, ..., 9) based on the values of all the other variables
#in the file (acidity, alcohol, pH, etc.)

In [8]:
# Training MLP Classifier
##############################
### Problem 1 & 3 combined ###
##############################

In [55]:
print('training model...')
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score

mlp = MLPClassifier()

hyperparameters = {'hidden_layer_sizes':[(50, ), (60, ),(50, 50), (60, 60)], 'activation':['relu', 'logistic']}
#                   testing 1 layer, with 50 or 60 neurons; then 2 layers, with 50 or 60 per layer
three_scoring = {'accuracy': make_scorer(accuracy_score),
           'precision': make_scorer(precision_score, average='weighted', zero_division=0),  # can use 'micro', 'macro', 'weighted', or None
           'recall': make_scorer(recall_score, average='weighted')}

from sklearn.exceptions import ConvergenceWarning
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning, module="sklearn")
    
    
# cross-validation
clf = GridSearchCV(mlp, hyperparameters, cv=5, scoring=three_scoring, refit='accuracy', verbose=4) #verbose = 4 gives more details
clf.fit(X_train, y_train)
print('done with training the model')
print("Best hyperparameters found on development set for MLP Classifier:")
print(clf.best_params_)
tuned_model_DT = clf.best_estimator_

# print(f'Decision tree has maximum depth {tuned_model_DT.tree_.max_depth}.')
y_pred = tuned_model_DT.predict(X_test)
# print( 'f1_score is')
# print( f1_score(y_test, y_pred, average='macro') )

training model...
Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV 1/5] END activation=relu, hidden_layer_sizes=(50,); accuracy: (test=0.487) precision: (test=0.493) recall: (test=0.487) total time=   1.7s
[CV 2/5] END activation=relu, hidden_layer_sizes=(50,); accuracy: (test=0.486) precision: (test=0.475) recall: (test=0.486) total time=   1.2s
[CV 3/5] END activation=relu, hidden_layer_sizes=(50,); accuracy: (test=0.524) precision: (test=0.478) recall: (test=0.524) total time=   1.6s
[CV 4/5] END activation=relu, hidden_layer_sizes=(50,); accuracy: (test=0.497) precision: (test=0.455) recall: (test=0.497) total time=   2.8s
[CV 5/5] END activation=relu, hidden_layer_sizes=(50,); accuracy: (test=0.488) precision: (test=0.500) recall: (test=0.488) total time=   1.6s
[CV 1/5] END activation=relu, hidden_layer_sizes=(60,); accuracy: (test=0.490) precision: (test=0.451) recall: (test=0.490) total time=   1.2s
[CV 2/5] END activation=relu, hidden_layer_sizes=(60,); accuracy

/project/cacds/apps/anaconda3/5.0.1/lib/python3.6/site-packages/sklearn/neural_network/_multilayer_perceptron.py:617: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  % self.max_iter, ConvergenceWarning)


[CV 4/5] END activation=relu, hidden_layer_sizes=(60,); accuracy: (test=0.520) precision: (test=0.495) recall: (test=0.520) total time=   3.2s
[CV 5/5] END activation=relu, hidden_layer_sizes=(60,); accuracy: (test=0.499) precision: (test=0.457) recall: (test=0.499) total time=   2.9s
[CV 1/5] END activation=relu, hidden_layer_sizes=(50, 50); accuracy: (test=0.520) precision: (test=0.485) recall: (test=0.520) total time=   3.6s
[CV 2/5] END activation=relu, hidden_layer_sizes=(50, 50); accuracy: (test=0.486) precision: (test=0.496) recall: (test=0.486) total time=   2.2s
[CV 3/5] END activation=relu, hidden_layer_sizes=(50, 50); accuracy: (test=0.513) precision: (test=0.497) recall: (test=0.513) total time=   3.9s
[CV 4/5] END activation=relu, hidden_layer_sizes=(50, 50); accuracy: (test=0.501) precision: (test=0.480) recall: (test=0.501) total time=   3.6s
[CV 5/5] END activation=relu, hidden_layer_sizes=(50, 50); accuracy: (test=0.466) precision: (test=0.435) recall: (test=0.466) tot

/project/cacds/apps/anaconda3/5.0.1/lib/python3.6/site-packages/sklearn/neural_network/_multilayer_perceptron.py:617: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  % self.max_iter, ConvergenceWarning)


[CV 1/5] END activation=logistic, hidden_layer_sizes=(50,); accuracy: (test=0.506) precision: (test=0.485) recall: (test=0.506) total time=   3.6s


...

/project/cacds/apps/anaconda3/5.0.1/lib/python3.6/site-packages/sklearn/neural_network/_multilayer_perceptron.py:617: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  % self.max_iter, ConvergenceWarning)


[CV 3/5] END activation=logistic, hidden_layer_sizes=(60, 60); accuracy: (test=0.547) precision: (test=0.506) recall: (test=0.547) total time=   6.4s


/project/cacds/apps/anaconda3/5.0.1/lib/python3.6/site-packages/sklearn/neural_network/_multilayer_perceptron.py:617: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  % self.max_iter, ConvergenceWarning)


[CV 4/5] END activation=logistic, hidden_layer_sizes=(60, 60); accuracy: (test=0.512) precision: (test=0.484) recall: (test=0.512) total time=   6.4s
[CV 5/5] END activation=logistic, hidden_layer_sizes=(60, 60); accuracy: (test=0.504) precision: (test=0.478) recall: (test=0.504) total time=   4.2s
done with training the model
Best hyperparameters found on development set for MLP Classifier:
{'activation': 'logistic', 'hidden_layer_sizes': (60,)}


/project/cacds/apps/anaconda3/5.0.1/lib/python3.6/site-packages/sklearn/neural_network/_multilayer_perceptron.py:617: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  % self.max_iter, ConvergenceWarning)


In [57]:
print('computing accuracy...')
y_predicted = clf.predict(X_test)  # make predictions using the validation data 
print('Accuracy:', accuracy_score(y_test, y_predicted))
print('Precision:', precision_score(y_test, y_predicted, average='macro', zero_division=0))
print('recall_score:', recall_score(y_test, y_predicted, average='macro', zero_division=0))
### Conclusion for problem 1 and problem 3 combined ###
# Here I used 3 types of scoring methods: accuracy_score, precision_score and recall_score
# Accuracy is 52.24%, Precision is 31.5% and recall is 26.22%
# accuracy_score 52.24%(straight forward, 52.24% is medeocre), 
# precision_score(false positive is not very costly, so that's fine) and 
# recall_score(low, cost of false negative is low so doesn't affect us too much)
# As we can see, the best architecture hyperparameter combination is: activation=logistic, hidden_layer=1, number of neurons=60
# From the activation being logistic regression. This is a very complicated choice because it requires expertise in the field. But in general it 
# works best for binary classification problems. 
# hidden layer is 1, therefore I can imagine there's not a lot of complexity to the data(low combined features). What I mean is it's a more simple linear combination of inputs
# instead of having a lot of features. IMO the more layers it require, the more complex the problem is (Like identifying Bob from an image)
# Number of neurons is 60: More neurons gives better accuracy, means we have a lot of parameters that will affect the final result. We need to adjust
# more parameters to achieve better learning. 

computing accuracy...
Accuracy: 0.5224489795918368
Precision: 0.3151161769571411
recall_score: 0.2622156993763523


In [31]:
##############################
### Problem 2 & 3 combined ###
##############################



# mlp = MLPClassifier(hidden_layer_sizes=(50,), # one hidden layer with 50 neurons
#                     activation = 'relu',  # ReLU is the default option
#                     # solver='sgd',  # default is Adam
#                     alpha=1e-4,  # regulariztion parameter, set to default=0.0001 (increase up to 1.0 for stronger regularization)
#                     learning_rate_init=.1 ,  # initial step-size for updating the weights, default is 0.001
#                     max_iter=10,  # number of epochs, default=200
#                     random_state=42,
#                     verbose=10, 
#                     )
print('training model...')
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score

mlp = MLPClassifier()
hyperparameters = {'activation':['logistic'], 'alpha': [0.0001, 0.00002], 'batch_size':['auto'], 'hidden_layer_sizes':[(60, )], 'learning_rate_init':[0.1, 0.2], 'solver': ['adam', 'sgd'] }
# I will vary alpha, learning_rate_init and solver. 

three_scoring = {'accuracy': make_scorer(accuracy_score),
           'precision': make_scorer(precision_score, average='weighted', zero_division=0),  # can use 'micro', 'macro', 'weighted', or None
           'recall': make_scorer(recall_score, average='weighted')}

from sklearn.exceptions import ConvergenceWarning
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning, module="sklearn")
    
    
# cross-validation
clf = GridSearchCV(mlp, hyperparameters, cv=5, scoring=three_scoring, refit='accuracy', verbose=4) #verbose = 4 gives more details
clf.fit(X_train, y_train)
print('done with training the model')
print("Best hyperparameters found on development set for MLP Classifier:")
print(clf.best_params_)
tuned_model_DT = clf.best_estimator_

# print(f'Decision tree has maximum depth {tuned_model_DT.tree_.max_depth}.')
y_pred = tuned_model_DT.predict(X_test)
# print( 'f1_score is')
# print( f1_score(y_test, y_pred, average='macro') )

training model...
Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV 1/5] END activation=logistic, alpha=0.0001, batch_size=auto, hidden_layer_sizes=(60,), learning_rate_init=0.1, solver=adam; accuracy: (test=0.452) precision: (test=0.204) recall: (test=0.452) total time=   0.7s
[CV 2/5] END activation=logistic, alpha=0.0001, batch_size=auto, hidden_layer_sizes=(60,), learning_rate_init=0.1, solver=adam; accuracy: (test=0.450) precision: (test=0.203) recall: (test=0.450) total time=   0.3s
[CV 3/5] END activation=logistic, alpha=0.0001, batch_size=auto, hidden_layer_sizes=(60,), learning_rate_init=0.1, solver=adam; accuracy: (test=0.450) precision: (test=0.203) recall: (test=0.450) total time=   0.4s
[CV 4/5] END activation=logistic, alpha=0.0001, batch_size=auto, hidden_layer_sizes=(60,), learning_rate_init=0.1, solver=adam; accuracy: (test=0.451) precision: (test=0.203) recall: (test=0.451) total time=   0.9s
[CV 5/5] END activation=logistic, alpha=0.0001, batch_size=aut

In [42]:
print('computing accuracy...')
y_predicted = clf.predict(X_test)  # make predictions using the validation data 
print('Accuracy:', accuracy_score(y_test, y_predicted))
print('Precision:', precision_score(y_test, y_predicted, average='macro', zero_division=0))
print('recall_score:', recall_score(y_test, y_predicted, average='macro', zero_division=0))

computing accuracy...
Accuracy: 0.44285714285714284
Precision: 0.14530812324929973
recall_score: 0.16968149420898562


In [44]:
##############################################
### Conclusion for problem 2 and problem 3 ###
##############################################

# Again, I used 3 different scoring methods: accuracy_score(straight forward, 44.29% is medeocre), 
# precision_score(false positive is not very costly, so that's fine) and 
# recall_score(low, cost of false negative is low so doesn't affect us too much), 
# The best optimizer parameters I used are: alpha = 0.0001, learning_rate_init=0.1 and solver=adam
# alpha = 0.0001 means we allow over-fitting, meaning we want the model to have higher complexity. 
# this indicates that wine quality is affected by a lot of variables
# learning_rate_init=0.1 is the same as the default value. means we don't want to go too fast into a particular W weight.
# Best solver is Adam instead of SGD. It makes sense because adam stores exponential decay average of past gradients, which will make the process 
# faster and higher accuracy. 

In [45]:
#######################
###### problem 4 ######
#######################

In [51]:
# Decision Tree
scoring = 'accuracy'

print('Tuning model...')
selected_model = DecisionTreeClassifier()
hyperparameters = {'max_depth':[5, 6, 7, 10, 20], 'criterion':['gini', 'entropy']}
clf = GridSearchCV(selected_model, hyperparameters, cv=5, scoring=scoring, verbose=4) #verbose = 4 gives more details

clf.fit(X_train, y_train)
print("Best hyperparameters found on development set for Decision Tree:")
print(clf.best_params_)
tuned_model_DT = clf.best_estimator_

print(f'Decision tree has maximum depth {tuned_model_DT.tree_.max_depth}.')
y_pred = tuned_model_DT.predict(X_test)
print( 'score is')
print( accuracy_score(y_test, y_pred) )

Tuning model...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END .......criterion=gini, max_depth=5;, score=0.528 total time=   0.0s
[CV 2/5] END .......criterion=gini, max_depth=5;, score=0.534 total time=   0.0s
[CV 3/5] END .......criterion=gini, max_depth=5;, score=0.550 total time=   0.0s
[CV 4/5] END .......criterion=gini, max_depth=5;, score=0.533 total time=   0.0s
[CV 5/5] END .......criterion=gini, max_depth=5;, score=0.526 total time=   0.0s
[CV 1/5] END .......criterion=gini, max_depth=6;, score=0.536 total time=   0.0s
[CV 2/5] END .......criterion=gini, max_depth=6;, score=0.527 total time=   0.0s
[CV 3/5] END .......criterion=gini, max_depth=6;, score=0.546 total time=   0.0s
[CV 4/5] END .......criterion=gini, max_depth=6;, score=0.507 total time=   0.0s
[CV 5/5] END .......criterion=gini, max_depth=6;, score=0.530 total time=   0.0s
[CV 1/5] END .......criterion=gini, max_depth=7;, score=0.550 total time=   0.0s
[CV 2/5] END .......criterion=gi

In [53]:
################################
### Conclusion for problem 4 ###
################################

In [54]:
# The score for Decision Tree with criterion=gini and max_denpth=20 has a score of 61.53% accuracy
# The neuron network MLPClassifier I used has a best score of 52.24% accuracy
# Therefore My neuron network MLPClassifier with my specified architecture choices and optimizeer parameters
# did not win the competition against Decision Tree. 
# This is a little bit disappointing, but not unexpected
# Neuron network usually work better when the problem is very complex with huge database. 
# The hyperparameter tuning might also be not as good as it can be, because I'm only exploring 6 hyperparameters in
# MLPClassifier. Therefore I accept the defeat for now. 